# 🔴 Hard: MLP Backpropagation (NumPy)

Write the **complete forward + backward pass** of an L-layer ReLU MLP with softmax cross-entropy —
no autograd, only NumPy. This is the exercise that makes `loss.backward()` stop being magic.

### Core Idea

Backprop is the chain rule executed in **reverse topological order**, reusing the values the forward
pass already computed. Two invariants make it hard to get wrong:

1. **Cache during the forward pass.** Every layer needs its input $A^{(l-1)}$ and its pre-activation
   $Z^{(l)}$ later. Recomputing them is the classic beginner's mistake — that is the memory/compute
   trade-off behind gradient checkpointing.
2. **Every gradient has the shape of the thing it differentiates.** `dW.shape == W.shape` always. If
   your shapes line up, your transposes are almost certainly right.

Seed the backward pass with the fused softmax-CE gradient, then walk backwards:

$$dZ^{(L)} = \frac{P - \mathbb{1}_y}{N}$$

$$dW^{(l)} = A^{(l-1)\top} dZ^{(l)}, \qquad db^{(l)} = \textstyle\sum_i dZ^{(l)}_i, \qquad dA^{(l-1)} = dZ^{(l)} W^{(l)\top}$$

$$dZ^{(l-1)} = dA^{(l-1)} \odot \mathbb{1}\!\left[Z^{(l-1)} > 0\right]$$

Read the three rules off the forward pass: `Z = A @ W + b` is a **product** (each factor's gradient
is the upstream gradient times the *other* factor), a **broadcast** (broadcasting forward ⇒ summing
backward, hence `db = dZ.sum(axis=0)`), and ReLU is a **gate** (it multiplies by a 0/1 mask forward,
so it multiplies by the same mask backward — a dead unit gets exactly zero gradient forever, which is
the "dying ReLU" problem in one line of algebra).

**Always gradient-check.** Compare against $(\mathcal{L}(w+\epsilon) - \mathcal{L}(w-\epsilon)) / 2\epsilon$;
with $\epsilon = 10^{-6}$ in float64 you should match to ~1e-8.

### Signature
```python
def mlp_loss_and_grads(X, labels, params):
    # X:      (N, d_in)
    # labels: (N,) integer class indices
    # params: list of (W, b) with W: (in, out), b: (out,)
    # returns: (loss, grads) where grads is a list of (dW, db) matching params
    ...
```

### Rules
- Pure **NumPy** — no PyTorch, no autograd
- ReLU on hidden layers, linear logits, softmax cross-entropy loss averaged over the batch
- `dW.shape == W.shape` and `db.shape == b.shape` for every layer

### Example
```
loss, grads = mlp_loss_and_grads(X, labels, params)
for (W, b), (dW, db) in zip(params, grads):   # one SGD step
    W -= lr * dW
    b -= lr * db
```

In [ ]:
import numpy as np

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def mlp_loss_and_grads(X, labels, params):
    # X: (N, d_in);  labels: (N,) int;  params: list of (W, b)
    # returns (loss, [(dW, db), ...])
    pass  # Replace this

In [ ]:
# 🧪 Test your implementation — train on a toy linearly-separable problem
np.random.seed(0)
X = np.random.randn(40, 4)
labels = (X[:, 0] + X[:, 1] > 0).astype(np.int64)
params = [(np.random.randn(4, 16) * 0.5, np.zeros(16)),
          (np.random.randn(16, 2) * 0.5, np.zeros(2))]

for step in range(300):
    loss, grads = mlp_loss_and_grads(X, labels, params)
    if step % 100 == 0:
        print(f"step {step:3d}  loss {float(loss):.4f}")
    for (W, b), (dW, db) in zip(params, grads):
        W -= 0.1 * dW
        b -= 0.1 * db

print(f"final     loss {float(loss):.4f}")

In [ ]:
# ✅ SUBMIT — Run this cell to check your solution
from torch_judge import check
check("numpy_mlp_backward")